# 2. Data Preprocessing

This notebook transforms the raw Facebook posts dataset into a validated numerical feature matrix suitable for dimensionality reduction and clustering.

## 2.1. Environment and Library Imports

In [18]:
%pip install pandas numpy scikit-learn joblib -q

Note: you may need to restart the kernel to use updated packages.


In [19]:
# Standard library imports
from pathlib import Path
import sys

# Third-party imports
import joblib
import numpy as np
import pandas as pd

In [20]:
current_directory = Path.cwd().resolve()
project_candidates = (current_directory, *current_directory.parents)

PROJECT_ROOT = next(
    (path for path in project_candidates if (path / "src").is_dir()),
    None,
)

if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate the project root directory.")

SRC_DIRECTORY = PROJECT_ROOT / "src"

if str(SRC_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SRC_DIRECTORY))

### 2.1.1. Custom Utilities

In [21]:
from data_collection_utils import (
    dataframe_overview,
    save_dataframe_csv,
    validate_dataframe_contract,
)
from data_preprocessing_utils import (
    add_cyclical_features,
    add_datetime_features,
    build_clustering_preprocessor,
    component_consistency_summary,
    validate_nonnegative_columns,
    validate_numeric_matrix,
)

## 2.2. Configuration

All paths and feature groups are defined in one place to make the workflow easier to maintain and audit.

In [22]:
RAW_DATA_PATH = (
    PROJECT_ROOT / "data" / "raw" / "facebook_live_sellers.csv"
)
CLEANED_DATA_PATH = (
    PROJECT_ROOT / "data" / "processed" / "facebook_posts_cleaned.csv"
)
MODEL_MATRIX_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "facebook_posts_model_matrix.csv"
)
PREPROCESSOR_PATH = PROJECT_ROOT / "artifacts" / "preprocessor.joblib"

EXPECTED_ROW_COUNT = 7_050
DATETIME_COLUMN = "status_published"
CATEGORICAL_COLUMNS = ["status_type"]
REACTION_COMPONENT_COLUMNS = [
    "num_likes",
    "num_loves",
    "num_wows",
    "num_hahas",
    "num_sads",
    "num_angrys",
]
ENGAGEMENT_COLUMNS = [
    "num_comments",
    "num_shares",
    *REACTION_COMPONENT_COLUMNS,
]
REQUIRED_COLUMNS = {
    DATETIME_COLUMN,
    *CATEGORICAL_COLUMNS,
    "num_reactions",
    *ENGAGEMENT_COLUMNS,
}

## 2.3. Raw Data Loading

This notebook consumes the immutable CSV artifact generated by `01_data_collection_and_understanding.ipynb`. It does not download the dataset again.

In [23]:
if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        "Raw dataset not found. Run notebook "
        "01_data_collection_and_understanding.ipynb first."
    )

raw_posts = pd.read_csv(RAW_DATA_PATH)

validate_dataframe_contract(
    raw_posts,
    required_columns=REQUIRED_COLUMNS,
)

if len(raw_posts) != EXPECTED_ROW_COUNT:
    raise ValueError(
        f"Expected {EXPECTED_ROW_COUNT:,} rows, "
        f"but found {len(raw_posts):,}."
    )

print(f"Dataset loaded from: {RAW_DATA_PATH}")
print(f"Shape: {raw_posts.shape}")
display(dataframe_overview(raw_posts))

Dataset loaded from: C:\Projetos\facebook_clusterizacao\data\raw\facebook_live_sellers.csv
Shape: (7050, 11)


,dtype,non_null,missing,missing_pct,unique
column,,,,,
status_type,str,7050,0,0.0,4
status_published,str,7050,0,0.0,6913
num_reactions,int64,7050,0,0.0,1067
num_comments,int64,7050,0,0.0,993
num_shares,int64,7050,0,0.0,501
num_likes,int64,7050,0,0.0,1044
num_loves,int64,7050,0,0.0,229
num_wows,int64,7050,0,0.0,65
num_hahas,int64,7050,0,0.0,42


## 2.4. Data Quality Assessment

Missing values, negative engagement counts, and exact duplicate feature rows are quantified before any transformation is applied.

In [24]:
numeric_columns = raw_posts.select_dtypes(include="number").columns
missing_value_count = int(raw_posts.isna().sum().sum())
negative_counts = validate_nonnegative_columns(
    raw_posts,
    numeric_columns,
)
duplicate_count = int(raw_posts.duplicated().sum())

if missing_value_count:
    raise ValueError(f"Found {missing_value_count} missing values.")

quality_summary = pd.Series(
    {
        "rows": len(raw_posts),
        "columns": raw_posts.shape[1],
        "missing_values": missing_value_count,
        "negative_numeric_values": int(negative_counts.sum()),
        "exact_duplicate_rows": duplicate_count,
        "duplicate_percentage": duplicate_count / len(raw_posts) * 100,
    },
    name="value",
)

display(quality_summary.to_frame())

,value
rows,7050.000000
columns,11.000000
missing_values,0.000000
negative_numeric_values,0.000000
exact_duplicate_rows,54.000000
duplicate_percentage,0.765957


The dataset contains 54 exact feature duplicates. They are preserved because the available feature table does not include the original post identifier or seller identifier. Identical feature values are therefore not sufficient evidence that the observations represent duplicated records. If identifiers become available later, this decision should be reassessed.

## 2.5. Temporal Feature Engineering

The original publication timestamp is parsed and expanded into interpretable calendar features. Month, weekday, and hour are also encoded as cyclical coordinates so adjacent boundary values remain close in the model space.

In [25]:
cleaned_posts = add_datetime_features(
    raw_posts,
    DATETIME_COLUMN,
    datetime_format="%m/%d/%Y %H:%M",
    parsed_column="published_at",
    feature_prefix="publication",
)

print(
    "Publication range: "
    f"{cleaned_posts['published_at'].min()} to "
    f"{cleaned_posts['published_at'].max()}"
)

Publication range: 2012-07-15 02:51:00 to 2018-06-13 01:12:00


### 2.5.1. Cyclical Encoding

In [26]:
model_features = add_cyclical_features(
    cleaned_posts,
    feature_periods={
        "publication_month": 12,
        "publication_weekday_number": 7,
        "publication_hour": 24,
    },
    offsets={"publication_month": 1},
)

In [27]:
CYCLICAL_COLUMNS = [
    "publication_month_sin",
    "publication_month_cos",
    "publication_weekday_number_sin",
    "publication_weekday_number_cos",
    "publication_hour_sin",
    "publication_hour_cos",
]

display(model_features[CYCLICAL_COLUMNS].head())

,publication_month_sin,publication_month_cos,publication_weekday_number_sin,publication_weekday_number_cos,publication_hour_sin,publication_hour_cos
0,1.0,6.123234e-17,-0.781831,0.623490,1.000000,6.123234e-17
1,1.0,6.123234e-17,-0.974928,-0.222521,-0.321439,9.469301e-01
2,1.0,6.123234e-17,-0.974928,-0.222521,0.997250,-7.410849e-02
3,1.0,6.123234e-17,-0.974928,-0.222521,0.605294,7.960020e-01
4,1.0,6.123234e-17,0.974928,-0.222521,0.771625,6.360782e-01


## 2.6. Reaction Consistency and Redundancy

`num_reactions` is compared with the sum of the individual reaction columns before the modeling feature set is defined.

In [28]:
reaction_consistency = component_consistency_summary(
    cleaned_posts,
    total_column="num_reactions",
    component_columns=REACTION_COMPONENT_COLUMNS,
)

display(reaction_consistency.to_frame())

,value
exact_matches,7041.0
different_rows,9.0
maximum_absolute_difference,4.0


`num_reactions` is retained in the cleaned dataset for interpretation but excluded from the clustering feature matrix. It is almost entirely derived from the individual reaction columns, so including both would give reaction activity duplicated weight in distance calculations.

## 2.7. Modeling Feature Matrix

A synthetic `record_id` is created only to align saved artifacts. It is never passed to PCA or a clustering algorithm.

In [29]:
cleaned_posts.insert(0, "record_id", np.arange(len(cleaned_posts)))

TEMPORAL_MODEL_COLUMNS = [
    "publication_year",
    "publication_month_sin",
    "publication_month_cos",
    "publication_weekday_number_sin",
    "publication_weekday_number_cos",
    "publication_hour_sin",
    "publication_hour_cos",
]
MODEL_INPUT_COLUMNS = [
    *ENGAGEMENT_COLUMNS,
    *CATEGORICAL_COLUMNS,
    *TEMPORAL_MODEL_COLUMNS,
]

model_input = model_features[MODEL_INPUT_COLUMNS].copy()
print(f"Model input shape: {model_input.shape}")

Model input shape: (7050, 16)


### 2.7.1. Preprocessing Pipelines

Engagement counts receive a `log1p` transformation before standardization because they are non-negative and strongly right-skewed. Temporal features are standardized separately. Post type is one-hot encoded without dropping a reference category, preserving a symmetric representation for distance-based methods.

In [30]:
preprocessor = build_clustering_preprocessor(
    log_scaled_columns=ENGAGEMENT_COLUMNS,
    standard_scaled_columns=TEMPORAL_MODEL_COLUMNS,
    categorical_columns=CATEGORICAL_COLUMNS,
)

In [31]:
model_matrix = preprocessor.fit_transform(model_input)
model_matrix.insert(
    0,
    "record_id",
    cleaned_posts["record_id"].to_numpy(),
)

display(model_matrix.head())

,record_id,num_comments,num_shares,num_likes,num_loves,num_wows,num_hahas,num_sads,num_angrys,publication_year,publication_month_sin,publication_month_cos,publication_weekday_number_sin,publication_weekday_number_cos,publication_hour_sin,publication_hour_cos,status_type_link,status_type_photo,status_type_status,status_type_video
0,0,1.719741,2.313934,1.188102,2.392021,1.613824,0.862031,1.711468,-0.229904,0.832651,1.341493,-0.102706,-1.100380,0.867571,1.073128,-0.288261,0.0,0.0,0.0,1.0
1,1,-0.951446,-0.641597,0.565108,-0.636607,-0.474460,-0.380746,-0.268881,-0.229904,0.832651,1.341493,-0.102706,-1.373478,-0.328929,-1.710015,1.060336,0.0,1.0,0.0,0.0
2,2,1.389190,1.512106,0.745911,1.428790,0.569682,0.862031,-0.268881,-0.229904,0.832651,1.341493,-0.102706,-1.373478,-0.328929,1.067336,-0.393805,0.0,0.0,0.0,1.0
3,3,-0.951446,-0.641597,0.388415,-0.636607,-0.474460,-0.380746,-0.268881,-0.229904,0.832651,1.341493,-0.102706,-1.373478,-0.328929,0.241820,0.845388,0.0,1.0,0.0,0.0
4,4,-0.951446,-0.641597,0.745911,0.901952,-0.474460,-0.380746,-0.268881,-0.229904,0.832651,1.341493,-0.102706,1.384215,-0.328929,0.592136,0.617628,0.0,1.0,0.0,0.0


## 2.8. Post-Transformation Validation

In [32]:
model_values = validate_numeric_matrix(
    model_matrix,
    excluded_columns=["record_id"],
    expected_row_count=len(cleaned_posts),
)

if not model_matrix["record_id"].equals(cleaned_posts["record_id"]):
    raise ValueError("The record identifiers are not aligned.")

print("Preprocessed model matrix validated successfully.")
print(f"Shape: {model_matrix.shape}")

Preprocessed model matrix validated successfully.
Shape: (7050, 20)


In [33]:
transformation_summary = pd.DataFrame(
    {
        "feature": model_values.columns,
        "dtype": model_values.dtypes.astype(str).to_numpy(),
        "mean": model_values.mean().to_numpy(),
        "std": model_values.std().to_numpy(),
        "minimum": model_values.min().to_numpy(),
        "maximum": model_values.max().to_numpy(),
    }
)

display(transformation_summary)

,feature,dtype,mean,std,minimum,maximum
0,num_comments,float64,0.000000e+00,1.000071,-0.951446,3.308502
1,num_shares,float64,-9.675476e-17,1.000071,-0.641597,3.675342
2,num_likes,float64,-1.612579e-17,1.000071,-2.402013,2.599679
3,num_loves,float64,8.062896e-17,1.000071,-0.636607,3.699400
4,num_wows,float64,1.451321e-16,1.000071,-0.474460,8.008276
5,num_hahas,float64,1.290063e-16,1.000071,-0.380746,8.696225
6,num_sads,float64,-5.644027e-17,1.000071,-0.268881,11.019982
7,num_angrys,float64,-3.225159e-17,1.000071,-0.229904,13.934798
8,publication_year,float64,-2.015724e-14,1.000071,-2.255297,0.832651
9,publication_month_sin,float64,7.458179e-17,1.000071,-1.441966,1.341493


## 2.9. Artifact Persistence

The cleaned dataset remains interpretable, while the model matrix contains only numerical features. The fitted preprocessor is stored so that the transformation can be reproduced without refitting.

In [34]:
cleaned_posts = cleaned_posts.drop(columns=[DATETIME_COLUMN])

cleaned_data_path = save_dataframe_csv(
    cleaned_posts,
    CLEANED_DATA_PATH,
)
model_matrix_path = save_dataframe_csv(
    model_matrix,
    MODEL_MATRIX_PATH,
)
PREPROCESSOR_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(preprocessor, PREPROCESSOR_PATH)

print(f"Cleaned dataset saved to: {cleaned_data_path}")
print(f"Model matrix saved to: {model_matrix_path}")
print(f"Fitted preprocessor saved to: {PREPROCESSOR_PATH}")

Cleaned dataset saved to: C:\Projetos\facebook_clusterizacao\data\processed\facebook_posts_cleaned.csv
Model matrix saved to: C:\Projetos\facebook_clusterizacao\data\processed\facebook_posts_model_matrix.csv
Fitted preprocessor saved to: C:\Projetos\facebook_clusterizacao\artifacts\preprocessor.joblib
